In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cài đặt vLLM (Lõi suy luận siêu tốc cho A100)
# Cài đặt FastAPI & Uvicorn & Ngrok (Cổng giao tiếp Web)
# Cài đặt LangChain & ChromaDB & Sentence-Transformers (Lõi RAG xử lý tài liệu)
!pip install -q vllm fastapi uvicorn pydantic pyngrok nest-asyncio langchain chromadb sentence-transformers pypdf docx2txt langchain-community langchain-text-splitters
!pip install -q langchain-experimental langchain-chroma
!pip install -U bitsandbytes accelerate transformers
!pip install python-dotenv

In [ ]:
import os
import torch
from sentence_transformers import CrossEncoder
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

# Nạp biến môi trường
load_dotenv('/content/drive/MyDrive/chatbotcolab/.env')
datapath = os.getenv('DATABASE_PATH')
if not datapath:
    print("⚠️ Cảnh báo: Không tìm thấy biến môi trường DATABASE_PATH, sẽ dùng thư mục mặc định.")
    datapath = "./my_vector_db"

class SmartKnowledgeBuilder:
    def __init__(self, db_path=datapath):
        self.db_path = db_path

        # FIX: Auto-detect GPU/CPU thay vì hardcode 'cuda'
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

        print("📥 [1/2] Đang tải mô hình Vector Embedding...")
        self.embedding_model = HuggingFaceEmbeddings(
            model_name="bkai-foundation-models/vietnamese-bi-encoder",
            model_kwargs={'device': device}
        )

        print("🕵️ [2/2] Đang tải mô hình Reranker (Thám tử chấm điểm)...")
        self.reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=2048)

        self.vector_db = None
        self._load_existing_db()

    def _load_existing_db(self):
        """Khôi phục lại DB nếu đã có trên ổ cứng"""
        if os.path.exists(self.db_path) and os.listdir(self.db_path):
            self.vector_db = Chroma(persist_directory=self.db_path, embedding_function=self.embedding_model)
            print(f"📁 Đã khôi phục Vector DB từ {self.db_path}")

    def process_and_save(self, file_path):
        """Chỉ tập trung vào Vector Search + Semantic Chunking"""
        _, ext = os.path.splitext(file_path)
        if ext.lower() == '.pdf':
            loader = PyPDFLoader(file_path)
        elif ext.lower() == '.txt':
            loader = TextLoader(file_path, encoding='utf-8')
        elif ext.lower() == '.docx':                    # FIX: Thêm hỗ trợ .docx
            loader = Docx2txtLoader(file_path)
        else:
            raise ValueError(f"Chưa hỗ trợ định dạng: {ext}")

        documents = loader.load()
        print("✂️ Đang cắt tài liệu bằng Semantic Chunking (Cắt theo ngữ nghĩa)...")

        text_splitter = SemanticChunker(
            self.embedding_model,
            breakpoint_threshold_type="percentile"
        )

        chunks = text_splitter.split_documents(documents)
        print(f"✅ Cắt xong! Hệ thống đã chia thành {len(chunks)} đoạn ngữ nghĩa.")

        print(f"🔄 Đang nhúng {len(chunks)} chunks vào ChromaDB...")
        self.vector_db = Chroma.from_documents(
            documents=chunks,
            embedding=self.embedding_model,
            persist_directory=self.db_path
        )
        print("✅ Đã hoàn tất xây dựng Smart Vector RAG!")

    def retrieve_context(self, query, top_k_vector=30, final_k=5):
        """
        Quy trình lấy thông tin chuẩn:
        1. Lấy 30 đoạn liên quan nhất (Vector Search).
        2. Dùng Reranker chấm điểm lại.
        3. Lấy 5 đoạn điểm cao nhất trả về cho LLM.
        """
        if not self.vector_db:
            return "Chưa có dữ liệu."

        raw_docs = self.vector_db.similarity_search(query, k=top_k_vector)
        chunk_texts = [doc.page_content for doc in raw_docs]

        if not chunk_texts:
            return "Không tìm thấy thông tin liên quan."

        pairs = [[query, text] for text in chunk_texts]
        scores = self.reranker.predict(pairs)

        paired_results = list(zip(chunk_texts, scores))
        paired_results.sort(key=lambda x: x[1], reverse=True)
        best_chunks = [item[0] for item in paired_results[:final_k]]

        return "\n---\n".join(best_chunks)


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from dotenv import load_dotenv


class AdvancedReasoningAgent:
    def __init__(self,hf_token):
        print("🧠 [LLM] Đang tải mô hình hạng nặng (14B-27B) với công nghệ Lượng tử hóa 4-bit...")
        self.hf_token = hf_token
        if not self.hf_token:
            print("⚠️ Cảnh báo: Không tìm thấy HF_TOKEN trong két sắt!")

        model_id = "google/gemma-2-27b-it"

        self.tokenizer = AutoTokenizer.from_pretrained(model_id, token=self.hf_token)

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            token=self.hf_token
        )

        # FIX: Xóa dòng set max_new_tokens toàn cục để tránh xung đột
        # self.model.generation_config.max_new_tokens = 4096  ← ĐÃ XÓA

        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer
        )
        print(f"✅ [LLM] Tải thành công mô hình {model_id} (Đã nén 4-bit)!")

    def _call_llm(self, prompt, temperature=0.1, max_new_tokens=2048):  # FIX: đổi tên param
        """Hàm helper để gọi pipeline cực kỳ "sạch", không có Warning"""
        messages = [{"role": "user", "content": prompt}]
        prompt_formatted = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        gen_kwargs = {
            "max_new_tokens": max_new_tokens,  # FIX: đúng tên tham số
            "pad_token_id": self.tokenizer.eos_token_id,
            "return_full_text": False
        }

        if temperature > 0.0:
            gen_kwargs["do_sample"] = True
            gen_kwargs["temperature"] = temperature
        else:
            gen_kwargs["do_sample"] = False

        outputs = self.pipe(prompt_formatted, **gen_kwargs)

        return outputs[0]["generated_text"].strip()

    # ==========================================
    # LOGIC 1: TREE OF THOUGHT (Tư duy cục bộ)
    # ==========================================
    def generate_thoughts(self, query, context, num_thoughts=5):
        prompt = f"Ngữ cảnh: {context}\nCâu hỏi: {query}\nHãy đưa ra {num_thoughts} hướng phân tích ngắn gọn và khác biệt để trả lời. Liệt kê bắt đầu bằng 'Hướng 1:', 'Hướng 2:'..."
        raw_text = self._call_llm(prompt, temperature=0.7)
        thoughts = [t.strip() for t in raw_text.split('\n') if len(t.strip()) > 10]
        return thoughts[:num_thoughts] if thoughts else [raw_text]

    def evaluate_thoughts_parallel(self, query, thoughts, context):
        best_thought = thoughts[0]
        best_score = -1

        for i, t in enumerate(thoughts):
            prompt = f"Ngữ cảnh: {context}\nCâu hỏi: {query}\nHướng giải quyết: '{t}'\nĐánh giá hướng này có đúng ngữ cảnh không. Chấm điểm (1-10). Chỉ xuất ra 1 con số nguyên."
            score_text = self._call_llm(prompt, temperature=0.0, max_new_tokens=1024)  # FIX: đúng param
            try:
                score = int(''.join(filter(str.isdigit, score_text)))
                if score > best_score:
                    best_score, best_thought = score, thoughts[i]
            except ValueError:
                pass
        return best_thought

    def self_reflect(self, query, best_thought, context):
        prompt = f"Bạn là AI cẩn thận. Ngữ cảnh: {context}\nCâu hỏi: {query}\nCâu trả lời nháp: {best_thought}\nNhiệm vụ: Sửa lại câu trả lời nháp sao cho mượt mà, không bịa đặt thông tin ngoài ngữ cảnh. Trả lời trực tiếp."
        return self._call_llm(prompt, temperature=0.1)

    # ==========================================
    # LOGIC 2: MAP-REDUCE (Đọc Toàn cục)
    # ==========================================
    def map_reduce_summarize(self, global_query, chunks_list=None):  # FIX: default=None thay vì 50
        if chunks_list is None:
            chunks_list = []

        print(f"🔄 [Map-Reduce] Bắt đầu đọc và tóm tắt {len(chunks_list)} phần tài liệu...")

        partial_summaries = []
        for i, chunk in enumerate(chunks_list):
            prompt_map = f"Đọc đoạn tài liệu sau:\n{chunk}\n\nHãy tóm tắt ngắn gọn các ý chính liên quan đến: '{global_query}'."
            summary = self._call_llm(prompt_map, temperature=0.1, max_new_tokens=2048)  # FIX: đúng param
            partial_summaries.append(summary)
            print(f"  -> Đã tóm tắt phần {i+1}/{len(chunks_list)}")

        print("🧠 [Map-Reduce] Đang tổng hợp các bản tóm tắt thành bài viết hoàn chỉnh...")
        combined_text = "\n".join(partial_summaries)

        prompt_reduce = f"Dưới đây là các bản tóm tắt từ nhiều phần của một tài liệu lớn:\n{combined_text}\n\nDựa trên các thông tin trên, hãy viết một câu trả lời hoàn chỉnh, mạch lạc cho yêu cầu: '{global_query}'."

        final_answer = self._call_llm(prompt_reduce, temperature=0.3, max_new_tokens=2048)  # FIX: đúng param
        return final_answer


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import ngrok

# --- KHỞI TẠO CÁC MODULE TOÀN CỤC ---
# Khởi tạo bên ngoài API để Model chỉ load 1 lần duy nhất vào VRAM
global_knowledge_base = SmartKnowledgeBuilder()
load_dotenv()
hf_token=os.getenv('HF_TOKEN')
if not hf_token:
    print("⚠️ Cảnh báo: Không tìm thấy HF_TOKEN trong két sắt!")
else:
    print(hf_token)
    global_agent = AdvancedReasoningAgent(hf_token)

# --- ĐỊNH NGHĨA API ---
app = FastAPI(title="Hệ thống Vistral ToT RAG API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class QueryRequest(BaseModel):
    question: str

class QueryResponse(BaseModel):
    final_answer: str
    thoughts_process: list
    context_used: str

@app.post("/api/v1/ask", response_model=QueryResponse)
async def ask_vistral(request: QueryRequest):
    try:
        query = request.question
        print(f"\n📨 [API] Nhận câu hỏi: '{query}'")

        # 1. Trích xuất kiến thức (RAG)
        context = global_knowledge_base.retrieve_context(query)

        # 2. Suy luận (ToT)
        thoughts = global_agent.generate_thoughts(query, context, num_thoughts=5)
        best_thought = global_agent.evaluate_thoughts_parallel(query, thoughts, context)

        # 3. Chốt đáp án (Self-Reflection)
        final_answer = global_agent.self_reflect(query, best_thought, context)

        print("✅ [API] Đã xử lý xong, trả kết quả cho Client.")
        return QueryResponse(final_answer=final_answer, thoughts_process=thoughts, context_used=context)

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
 # Tự động tìm và đọc file .env
load_dotenv()
doc_path = os.getenv('DOC_PATH')
file_path_cua_ban = doc_path
global_knowledge_base.process_and_save(file_path_cua_ban)


In [ ]:
import uvicorn
from pyngrok import ngrok
import nest_asyncio
import asyncio

ngrok_token = os.getenv('NGROK_TOKEN')
# ĐIỀN NGROK AUTHTOKEN CỦA BẠN VÀO ĐÂY
NGROK_AUTH_TOKEN = ngrok_token
try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok.kill() # Dọn dẹp kết nối cũ để không bị kẹt port

    public_url = ngrok.connect(8000).public_url
    print("\n" + "="*60)
    print(f"🟢 MÁY CHỦ ĐÃ SẴN SÀNG!")
    print(f"🔗 URL Của API (Dùng cho Frontend): {public_url}/api/v1/ask")
    print(f"📚 Bảng điều khiển Test API (Swagger): {public_url}/docs")
    print("="*60 + "\n")

    # Bơm nest_asyncio để Colab cho phép các tác vụ bất đồng bộ linh hoạt hơn
    nest_asyncio.apply()

    # 🛠 BÍ QUYẾT: Thay vì uvicorn.run(), ta dùng uvicorn.Server và await nó
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)

    # Từ khóa 'await' báo hiệu: Hãy chạy server này bên trong Event Loop hiện tại của Colab
    await server.serve()

except Exception as e:
    print(f"❌ Có lỗi khi khởi động máy chủ: {e}")